# 01. Queue: Fundamentals

A queue is **FIFO**: First In, First Out. Add at the **back**, remove from the **front**.

**Learning order:** REAL LIFE → OPERATIONS → WHY NOT list.pop(0) → deque → CIRCULAR QUEUE → QUEUE WITH STACKS → RECENT COUNTER → INTERVIEW THINKING

## 1. Real-life examples

- **Ticket counter / ATM line:** the first person who came is served first.
- **Printer queue:** documents print in the order they were sent.
- **Task scheduling, message queues (Kafka, SQS), customer support tickets.**
- **BFS** (next notebooks) uses a queue to explore level by level.

**Recognition clue:** "process in the order of arrival", "level by level", "sliding window of recent events".

## 2. Operations

| Operation | `collections.deque` | Cost |
|---|---|---|
| enqueue (add back) | `q.append(x)` | O(1) |
| dequeue (remove front) | `q.popleft()` | O(1) |
| peek front / back | `q[0]` / `q[-1]` | O(1) |
| empty | `not q` | O(1) |

**Never use `list.pop(0)`:** it shifts every element, so it is **O(n)**. A BFS with `pop(0)` becomes O(n²). Use `deque`.

In [1]:
from collections import deque

q = deque()
q.append(1); q.append(2); q.append(3)
print(q.popleft(), q[0], len(q))   # 1 2 2

1 2 2


## 3. Circular Queue (fixed size)

Use an array of size `k`. Keep `head` (front index) and `count`. The back position is `(head + count) % k`. The modulo makes the array "wrap around", so freed slots at the front are reused.

In [2]:
class CircularQueue:
    def __init__(self, k):
        self.data = [0] * k
        self.k, self.head, self.count = k, 0, 0

    def enqueue(self, x):
        if self.count == self.k:
            return False
        self.data[(self.head + self.count) % self.k] = x
        self.count += 1
        return True

    def dequeue(self):
        if self.count == 0:
            return False
        self.head = (self.head + 1) % self.k
        self.count -= 1
        return True

    def front(self):
        return -1 if self.count == 0 else self.data[self.head]

    def rear(self):
        return -1 if self.count == 0 else self.data[(self.head + self.count - 1) % self.k]

cq = CircularQueue(3)
print(cq.enqueue(1), cq.enqueue(2), cq.enqueue(3), cq.enqueue(4))   # True True True False
print(cq.rear(), cq.count == 3)   # 3 True
cq.dequeue()
print(cq.enqueue(4), cq.rear())   # True 4

True True True False
3 True
True 4


## 4. Queue using two stacks (amortized O(1))

- `push` → always into `inbox`.
- `pop`/`peek` → if `outbox` is empty, **move everything** from `inbox` to `outbox` (this reverses the order, so the oldest is on top). Then pop from `outbox`.

Each element moves at most once between stacks, so the cost is **amortized O(1)**.

In [3]:
class MyQueue:
    def __init__(self):
        self.inbox, self.outbox = [], []

    def push(self, x):
        self.inbox.append(x)

    def _shift(self):
        if not self.outbox:
            while self.inbox:
                self.outbox.append(self.inbox.pop())

    def pop(self):
        self._shift()
        return self.outbox.pop()

    def peek(self):
        self._shift()
        return self.outbox[-1]

    def empty(self):
        return not self.inbox and not self.outbox

mq = MyQueue()
mq.push(1); mq.push(2)
print(mq.peek(), mq.pop(), mq.empty())   # 1 1 False

1 1 False


## 5. Stack using one queue

After pushing `x`, **rotate** the queue `len - 1` times so `x` moves to the front. Then `popleft()` behaves like a stack pop. Push is O(n), pop is O(1).

In [4]:
class MyStack:
    def __init__(self):
        self.q = deque()

    def push(self, x):
        self.q.append(x)
        for _ in range(len(self.q) - 1):
            self.q.append(self.q.popleft())

    def pop(self):
        return self.q.popleft()

    def top(self):
        return self.q[0]

ms = MyStack()
ms.push(1); ms.push(2); ms.push(3)
print(ms.pop(), ms.top())   # 3 2

3 2


## 6. Number of Recent Calls

Count requests in the last 3000 ms. Keep timestamps in a deque. Before answering, remove timestamps that are too old from the **front**. Each timestamp is added and removed once → O(1) amortized.

In [5]:
class RecentCounter:
    def __init__(self):
        self.q = deque()

    def ping(self, t):
        self.q.append(t)
        while self.q[0] < t - 3000:
            self.q.popleft()
        return len(self.q)

rc = RecentCounter()
print([rc.ping(t) for t in [1, 100, 3001, 3002]])   # [1, 2, 3, 3]

[1, 2, 3, 3]


## 7. Interview thinking

**Signals:** "first come first served", "level order", "recent events in a window", "simulate a line".

**Say:** "I use a deque so both enqueue and dequeue are O(1). A list's `pop(0)` would be O(n)."

**Common mistakes:** `list.pop(0)`, off-by-one in circular index math, confusing `count` and `size`, not checking empty before `popleft()` (raises `IndexError`).

## Quick Revision

- FIFO. `deque.append` / `popleft`, both O(1). Never `list.pop(0)`.
- Circular queue: `(head + count) % k`.
- Queue from two stacks: inbox/outbox, amortized O(1).
- Recent events: drop old timestamps from the front.